In [3]:
import os
import numpy as np
import pandas as pd
import librosa
from scipy.stats import skew, kurtosis

In [4]:
def extract_cough_features(file_path, sr=22050):
    y, sr = librosa.load(file_path, sr=sr)
    y = librosa.util.normalize(y)

    stft = np.abs(librosa.stft(y))

    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)
    mfcc_delta = librosa.feature.delta(mfcc)

    centroid = librosa.feature.spectral_centroid(S=stft, sr=sr)
    bandwidth = librosa.feature.spectral_bandwidth(S=stft, sr=sr)
    rolloff = librosa.feature.spectral_rolloff(S=stft, sr=sr)
    flatness = librosa.feature.spectral_flatness(S=stft)

    zcr = librosa.feature.zero_crossing_rate(y)
    rms = librosa.feature.rms(y=y)

    skewness = skew(y)
    kurt = kurtosis(y)

    features = []

    features.extend(np.mean(mfcc, axis=1))
    features.extend(np.std(mfcc, axis=1))

    features.extend(np.mean(mfcc_delta, axis=1))
    features.extend(np.std(mfcc_delta, axis=1))

    features.append(np.mean(centroid))
    features.append(np.std(centroid))

    features.append(np.mean(bandwidth))
    features.append(np.std(bandwidth))

    features.append(np.mean(rolloff))
    features.append(np.std(rolloff))

    features.append(np.mean(flatness))
    features.append(np.std(flatness))

    features.append(np.mean(zcr))
    features.append(np.std(zcr))

    features.append(np.mean(rms))
    features.append(np.std(rms))

    features.append(skewness)
    features.append(kurt)

    return np.array(features)


def safe_extract_cough_features(file_path, sr=22050):
    try:
        return extract_cough_features(file_path, sr=sr)
    except Exception as e:
        print(f"Error processing {file_path}: {e}")
        return np.full(66, np.nan)

In [5]:
def get_feature_column_names():
    cols = []

    for i in range(1, 14):
        cols.append(f"mfcc_{i}_mean")

    for i in range(1, 14):
        cols.append(f"mfcc_{i}_std")

    for i in range(1, 14):
        cols.append(f"mfcc_delta_{i}_mean")

    for i in range(1, 14):
        cols.append(f"mfcc_delta_{i}_std")

    cols.extend([
        "spectral_centroid_mean", "spectral_centroid_std",
        "spectral_bandwidth_mean", "spectral_bandwidth_std",
        "spectral_rolloff_mean", "spectral_rolloff_std",
        "spectral_flatness_mean", "spectral_flatness_std",
        "zcr_mean", "zcr_std",
        "rms_mean", "rms_std",
        "skewness", "kurtosis"
    ])

    return cols

In [6]:
input_csv = r"final-working-dataset-11391.csv"
df = pd.read_csv(input_csv)

print("Loaded:", df.shape)
print(df["cough_path"].head(3))

Loaded: (11391, 9)
0    E:\first_iterations\result\00ad476fba37b_cough...
1    E:\first_iterations\result\00ad476fba37b_cough...
2    E:\first_iterations\result\00ad476fba37b_cough...
Name: cough_path, dtype: object


In [7]:
base_dir = r"D:\first_iterations\result-clean\clean"

df["cough_path"] = df["cough_path"].apply(
    lambda x: os.path.join(base_dir, os.path.basename(str(x)))
)

In [8]:
print(df["cough_path"].head(5).to_list())

print(df["cough_path"].apply(os.path.exists).value_counts())

['D:\\first_iterations\\result-clean\\clean\\00ad476fba37b_cough_13.wav', 'D:\\first_iterations\\result-clean\\clean\\00ad476fba37b_cough_15.wav', 'D:\\first_iterations\\result-clean\\clean\\00ad476fba37b_cough_16.wav', 'D:\\first_iterations\\result-clean\\clean\\00ad476fba37b_cough_18.wav', 'D:\\first_iterations\\result-clean\\clean\\00ad476fba37b_cough_2.wav']
cough_path
True    11391
Name: count, dtype: int64


In [9]:
df = df[df["cough_path"].apply(os.path.exists)].copy()

print("After filtering:", df.shape)

After filtering: (11391, 9)


In [10]:
test_path = df["cough_path"].iloc[0]
print("Testing:", test_path)

fv = extract_cough_features(test_path)

print("Feature shape:", fv.shape)
print("First 5 values:", fv[:5])

Testing: D:\first_iterations\result-clean\clean\00ad476fba37b_cough_13.wav


c:\Users\ishra\anaconda3\envs\ml\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Feature shape: (66,)
First 5 values: [-147.53507996   72.21966553  -86.52467346   33.40273666  -49.34981155]


In [11]:
feature_list = []

for i, path in enumerate(df["cough_path"]):
    if i < 5:
        print("Processing:", path)

    feature_list.append(safe_extract_cough_features(path))

Processing: D:\first_iterations\result-clean\clean\00ad476fba37b_cough_13.wav
Processing: D:\first_iterations\result-clean\clean\00ad476fba37b_cough_15.wav
Processing: D:\first_iterations\result-clean\clean\00ad476fba37b_cough_16.wav
Processing: D:\first_iterations\result-clean\clean\00ad476fba37b_cough_18.wav
Processing: D:\first_iterations\result-clean\clean\00ad476fba37b_cough_2.wav


In [12]:
feature_array = np.vstack(feature_list)
print("Feature array shape:", feature_array.shape)

Feature array shape: (11391, 66)


In [13]:
feature_df = pd.DataFrame(
    feature_array,
    columns=get_feature_column_names()
)

print(feature_df.head())

   mfcc_1_mean  mfcc_2_mean  mfcc_3_mean  mfcc_4_mean  mfcc_5_mean  \
0  -147.535080    72.219666   -86.524673    33.402737   -49.349812   
1  -187.801849    64.501816   -80.451691    29.111990   -65.727478   
2  -276.585907    61.476242     4.864369    91.261169   -20.212484   
3  -168.387268    64.773430   -68.518677    27.705652   -59.638706   
4  -225.394714    59.819851   -82.262077     9.598791   -48.341011   

   mfcc_6_mean  mfcc_7_mean  mfcc_8_mean  mfcc_9_mean  mfcc_10_mean  ...  \
0    22.672956   -64.984940     8.177164     0.026583    -19.660343  ...   
1    25.034943   -91.621193     7.484174     9.475457    -21.164030  ...   
2     2.835943   -94.483696     8.761132    -8.397576    -37.812626  ...   
3    21.916605   -86.817223     8.105147     9.937062    -15.004102  ...   
4    44.027748   -87.274643    -0.824641    14.847258    -31.587631  ...   

   spectral_rolloff_mean  spectral_rolloff_std  spectral_flatness_mean  \
0            5381.570435            607.426844  

In [15]:
final_df = pd.concat(
    [df.reset_index(drop=True), feature_df.reset_index(drop=True)],
    axis=1
)

print("Final shape:", final_df.shape)

Final shape: (11391, 75)


In [16]:
output_csv = r"D:\first_iterations\cough_features_66_final.csv"

final_df.to_csv(output_csv, index=False)

print("Saved to:", output_csv)

Saved to: D:\first_iterations\cough_features_66_final.csv
